# M03：时间序列与技术分析

本 Notebook 涵盖量化交易中时间序列分析的核心实战环节：
1. **平稳性与时序建模**：对数收益率、单位根检验(ADF)、ACF/PACF 绘图、ARIMA 模型拟合。
2. **协整与配对交易**：寻找具备协整关系（长期均值回归）的股票对（如 KO / PEP），演示 Engle-Granger 两步法。
3. **波动率建模**：使用 GARCH(1,1) 模型拟合美股大盘指数（如 SPY）并预测未来波动率。

我们使用 `yfinance` 直接拉取演示数据，依赖 `statsmodels` 和 `arch` 库进行统计建模。

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from statsmodels.tsa.stattools import adfuller, acf, pacf, coint
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from arch import arch_model

# 绘图设置
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. 平稳性与 ARIMA 基础建模

**知识点**：金融资产的价格序列通常是**非平稳**的（存在随机游走或趋势），直接对价格进行回归容易导致“伪回归”。通常我们需要取**对数收益率**（Log Returns），将其转换为平稳序列。

In [ ]:
# 获取 SPY (标普500 ETF) 过去 3 年的数据
spy = yf.download('SPY', start='2021-01-01', end='2024-01-01')['Close']
spy = spy.dropna()

# 计算对数收益率: log(P_t / P_{t-1})
log_returns = np.log(spy / spy.shift(1)).dropna()

fig, axes = plt.subplots(2, 1, figsize=(12, 10))
spy.plot(ax=axes[0], title='SPY Price (Non-stationary)')
log_returns.plot(ax=axes[1], title='SPY Log Returns (Stationary)')
plt.tight_layout()
plt.show()

In [ ]:
def test_stationarity(timeseries, title=""):
    """执行 ADF 检验，判断序列是否平稳"""
    print(f"--- ADF Test: {title} ---")
    result = adfuller(timeseries, autolag='AIC')
    print(f'ADF Statistic: {result[0]:.4f}')
    print(f'p-value: {result[1]:.4f}')
    print('Critical Values:')
    for key, value in result[4].items():
        print(f'   {key}: {value:.4f}')
    
    if result[1] <= 0.05:
        print("结论: 拒绝原假设，序列是【平稳】的\n")
    else:
        print("结论: 无法拒绝原假设，序列是【非平稳】的\n")

# 价格 vs 收益率 平稳性对比
test_stationarity(spy, "SPY Price")
test_stationarity(log_returns, "SPY Log Returns")

### ACF/PACF 绘图与 ARIMA 模型拟合
通过 ACF（自相关）和 PACF（偏自相关）图来判断 AR(p) 和 MA(q) 的阶数。

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
plot_acf(log_returns, lags=40, ax=axes[0], title="ACF of SPY Log Returns")
plot_pacf(log_returns, lags=40, ax=axes[1], title="PACF of SPY Log Returns")
plt.show()

# 大盘的收益率通常非常接近白噪声，ACF/PACF 几乎没有显著的滞后截尾。
# 我们强行拟合一个 ARIMA(1,0,1) 即 ARMA(1,1) 来演示流程
model = ARIMA(log_returns, order=(1, 0, 1))
results = model.fit()
print(results.summary())

# 残差分析 (如果模型拟合得好，残差应该接近白噪声)
residuals = pd.DataFrame(results.resid)
fig, axes = plt.subplots(1, 2, figsize=(15, 4))
residuals.plot(ax=axes[0], title="Residuals")
residuals.plot(kind='kde', ax=axes[1], title="Density")
plt.show()

## 2. 协整检验与配对交易 (Pairs Trading)

**理论**：如果两个非平稳的价格序列 $X_t$ 和 $Y_t$ 之间存在某个线性组合 $Y_t - \beta X_t$ 是平稳的，我们就称 $X_t$ 和 $Y_t$ 具有**协整关系**。
这构成了统计套利中配对交易的基础：当价差偏离长期均值时，做空被高估的一方，做多被低估的一方。

我们测试经典配对：可口可乐 (KO) 与百事可乐 (PEP)。

In [ ]:
tickers = ['KO', 'PEP']
data = yf.download(tickers, start='2020-01-01', end='2024-01-01')['Close']
data = data.dropna()

# 标准化后查看走势（同为饮料巨头，走势往往趋同）
normalized_data = data / data.iloc[0]
normalized_data.plot(title="KO vs PEP (Normalized Price)")
plt.show()

In [ ]:
# Engle-Granger 两步法检验协整
score, pvalue, _ = coint(data['KO'], data['PEP'])
print(f"Cointegration test p-value: {pvalue:.4f}")
if pvalue < 0.05:
    print("结论: 存在显著的协整关系！可以进行配对交易。\n")
else:
    print("结论: 在此时间段内协整关系不显著。\n")

# 可视化价差 (Spread) 和 Z-Score
spread = data['KO'] - data['PEP']
zscore = (spread - spread.mean()) / spread.std()

plt.figure(figsize=(12, 6))
zscore.plot(label='Z-Score')
plt.axhline(0, color='black')
plt.axhline(2, color='red', linestyle='--')
plt.axhline(-2, color='green', linestyle='--')
plt.title("Z-Score of Spread (KO - PEP)")
plt.legend()
plt.show()

## 3. 波动率建模：GARCH(1,1)

金融时序的方差通常不是恒定的，存在**波动率聚集（Volatility Clustering）**现象：大波动伴随大波动，小波动伴随小波动。
GARCH(1,1) 是最经典的预测波动率的模型，公式核心思想是：今天的方差依赖于昨天的方差和昨天的残差平方。

In [ ]:
# GARCH 模型通常需要收益率乘以 100 以放大数值，有助于求解器收敛
returns_100 = log_returns * 100

# 构建 GARCH(1,1) 模型 (均值方程设为 Constant)
am = arch_model(returns_100, vol='Garch', p=1, q=1, dist='Normal')
res = am.fit(disp='off')
print(res.summary())

# 提取模型拟合的条件波动率 (Conditional Volatility)
fig, ax = plt.subplots(figsize=(12, 5))
returns_100.plot(ax=ax, alpha=0.5, label='Daily Returns (%)', color='gray')
res.conditional_volatility.plot(ax=ax, color='red', label='Conditional Volatility (GARCH 1,1)')
plt.title("SPY Returns and GARCH Volatility")
plt.legend()
plt.show()

# 预测未来 20 日的方差
forecasts = res.forecast(horizon=20)
pred_var = forecasts.variance.iloc[-1]
pred_vol = np.sqrt(pred_var) # 转换为标准差(%)

plt.figure(figsize=(8, 4))
plt.plot(range(1, 21), pred_vol, marker='o', linestyle='-')
plt.title("Forecasted Volatility for next 20 days")
plt.xlabel("Days Ahead")
plt.ylabel("Predicted Volatility (%)")
plt.show()